In [ ]:
# ⚠️ CRITICAL: Must run this FIRST and ONLY ONCE!
# This cell completely removes torchvision to prevent circular import errors
import subprocess
import sys
import os

print("⚠️  Step 1: Uninstalling torchvision completely...")
result = subprocess.run(
    ["pip", "uninstall", "-y", "torchvision"],
    capture_output=True,
    text=True,
    timeout=60
)
print(f"   {result.stdout.split(chr(10))[0]}")

print("\n✅ Step 2: Setting environment variables...")
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("   ✅ All environment variables set")

print("\n✅ Step 3: Installing import hook...")
# Remove any cached torchvision modules
modules_to_remove = [name for name in list(sys.modules.keys()) if 'torchvision' in name.lower()]
for module_name in modules_to_remove:
    del sys.modules[module_name]
print(f"   ✅ Removed {len(modules_to_remove)} cached torchvision modules")

# Block future imports
class BlockTorchvision:
    def find_module(self, fullname, path=None):
        if 'torchvision' in fullname.lower():
            raise ImportError("torchvision is permanently disabled")
        return None

sys.meta_path.insert(0, BlockTorchvision())
print("   ✅ Import hook installed")

print("\n" + "="*70)
print("✅ ALL TORCHVISION BLOCKS ACTIVATED")
print("="*70)
print("\n⚠️  IMPORTANT: If you see torchvision-related errors, RESTART the kernel")
print("   and run this cell again as the VERY FIRST cell.\n")


In [ ]:
# 导入必需的库
import os
import json
import subprocess
import numpy as np
import torch
from pathlib import Path

print("✅ Core imports loaded: os, json, subprocess, numpy, torch")


# 🚀 GIS代码生成模型训练 - 步骤级指令 (Step-Level CodeLlama)

本Notebook在Google Colab上训练GIS代码生成模型（**步骤级 + CodeLlama + 关键词权重**）

## 📌 步骤级训练 vs 文件级训练

| 维度 | 步骤级 | 文件级 |
|------|--------|--------|
| **训练数据** | step_level_instructions.jsonl (40,209条) | hierarchical_training_data.json (4,013文件) |
| **输入** | 上下文 + 步骤参数 | 完整的高层指令 |
| **输出** | 单步动作指令 ("Open E MS Kabel") | 完整JSON工作流 |
| **优势** | 数据量大、更细粒度、可应用权重损失 | 全局视角、生成完整工作流 |
| **权重** | ✅ 支持 keyword_weights | ❌ 不支持 |

## 🎯 本Notebook的创新点

1. **关键词权重损失** - 根据keyword_weights (action=3.0, object=2.0, context=1.5) 加权损失
2. **复合对象识别** - "E MS Kabel" 作单个词组识别 (52个已知对象)
3. **~20k样本** - step_level_instructions.jsonl的50%用于训练
4. **简洁格式** - 输出 "Open MS Kabel" (不含Step X/Y前缀)
5. **内存优化** - Batch Size 2 + Gradient Checkpointing 适配T4 GPU

**预计时间**: 1.5-2小时 (T4) / 30分钟-1小时 (A100)

---

## 📋 步骤1：环境设置

In [ ]:
# 检查GPU
!nvidia-smi

In [ ]:
# 安装依赖（约3-5分钟）
print("📦 Installing dependencies...")

# 先锁定关键基础包（避免自动升级）
!pip install -q torch==2.9.0 --no-deps
!pip install -q fsspec==2024.3.1
!pip install -q numpy==2.0.2 --no-deps

# 安装主要训练库（指定兼容版本）
!pip install -q transformers==4.46.0
!pip install -q peft==0.13.0
!pip install -q datasets==2.19.0
!pip install -q "accelerate>=1.0.0"
!pip install -q sentencepiece==0.2.0
!pip install -q tqdm
!pip install -q huggingface-hub==0.26.0

print("✅ Core dependencies installed! If running in Colab, restart runtime after this cell.")

## 💾 步骤2：挂载Google Drive（保存模型）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 创建输出目录
!mkdir -p /content/drive/MyDrive/gis-models
print("✅ Google Drive mounted!")

## 📂 步骤3：加载步骤级数据

In [ ]:
import os

# 首先，确保我们回到根目录，避免在错误的位置克隆
%cd /content/

# 删除可能存在的旧仓库副本，确保全新的克隆
!rm -rf gis-code-ai

# 克隆您的GitHub仓库
GITHUB_REPO_URL = "https://github.com/rockyistt/gis-code-ai" # 用户提供的URL

print(f"📦 克隆仓库: {GITHUB_REPO_URL}...")
!git clone {GITHUB_REPO_URL}

# 检查是否成功克隆并进入目录
if os.path.exists('gis-code-ai'):
    print("✅ 仓库克隆成功！")
    %cd gis-code-ai
    print(f"📍 当前工作目录已切换到: {os.getcwd()}")
    print("📂 目录内容: ")
    !ls -F

    # 再次检查数据文件路径
    expected_instructions_file = 'data/processed/step_level_instructions.jsonl'
    expected_data_file = 'data/processed/step_level_data.jsonl'

    if os.path.exists(expected_instructions_file) and os.path.exists(expected_data_file):
        print(f"✅ 已找到数据文件: {expected_instructions_file} 和 {expected_data_file}")
        print("   现在您可以尝试重新运行数据加载单元 (cell `XraxMOdNVGuh`)。")
    else:
        print("❌ 警告: 克隆后数据文件仍未找到。请检查您的GitHub仓库中 `data/processed/` 路径下是否包含 `step_level_instructions.jsonl` 和 `step_level_data.jsonl`。")
        print(f"   当前 {os.getcwd()}/data/ 目录内容:")
        !ls -F data/
else:
    print("❌ 仓库克隆失败，请检查您的GitHub仓库URL或权限。")

print("--------------------------------------------------")
print("克隆完成后，请运行 '步骤3：加载步骤级数据' 部分的代码单元以加载数据。")

In [ ]:
import os
import json
import sys
import numpy as np
import random
from pathlib import Path

print("="*70)
print("🔍 第一步：检查和加载数据文件")
print("="*70)

# 确保在正确的目录
if os.path.exists('/content/gis-code-ai'):
    os.chdir('/content/gis-code-ai')
elif os.path.exists('gis-code-ai'):
    os.chdir('gis-code-ai')

print(f"\n📍 当前工作目录: {os.getcwd()}\n")

# ============================================================
# 第1部分：检查数据文件
# ============================================================

print("📋 检查数据文件...\n")

SOURCE_FILES = {
    '✅ 步骤级指令': 'data/processed/step_level_instructions.jsonl',
    '✅ 步骤级数据': 'data/processed/step_level_data.jsonl',
}

files_status = {}
present_files = {}

for desc, filepath in SOURCE_FILES.items():
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                lines = sum(1 for _ in f)
            print(f"{desc} ✅")
            print(f"   📁 {filepath}")
            print(f"   💾 大小: {size_mb:.1f} MB | 📊 行数: {lines:,}\n")
            
            files_status[filepath] = 'OK'
            present_files[filepath] = size_mb
            
        except Exception as e:
            print(f"{desc} ⚠️")
            print(f"   📁 {filepath}")
            print(f"   ⚠️ 读取失败: {e}\n")
            files_status[filepath] = 'ERROR'
    else:
        print(f"{desc} ❌")
        print(f"   📁 {filepath}")
        print(f"   ❌ 必需但未找到\n")
        files_status[filepath] = 'MISSING'

# ============================================================
# 第2部分：加载数据文件
# ============================================================

print("="*70)
print("📂 第二步：加载数据")
print("="*70 + "\n")

# 加载step_level_instructions
all_instructions = []
print("1️⃣ 加载步骤级指令...")
try:
    with open('data/processed/step_level_instructions.jsonl', 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if line.strip():
                try:
                    item = json.loads(line)
                    all_instructions.append(item)
                except json.JSONDecodeError as e:
                    print(f"   ⚠️  第 {line_num} 行解析失败: {e}")
    print(f"   ✅ 已加载: {len(all_instructions):,} 条指令\n")
except Exception as e:
    print(f"   ❌ 加载失败: {e}\n")
    sys.exit(1)

# 加载step_level_data
all_data = []
print("2️⃣ 加载步骤级数据...")
try:
    with open('data/processed/step_level_data.jsonl', 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if line.strip():
                try:
                    item = json.loads(line)
                    all_data.append(item)
                except json.JSONDecodeError as e:
                    print(f"   ⚠️  第 {line_num} 行解析失败: {e}")
    print(f"   ✅ 已加载: {len(all_data):,} 条数据\n")
except Exception as e:
    print(f"   ❌ 加载失败: {e}\n")
    sys.exit(1)

# 验证数据一致性
if len(all_instructions) != len(all_data):
    print(f"❌ 错误：数据数量不匹配！")
    print(f"   指令: {len(all_instructions)}")
    print(f"   数据: {len(all_data)}")
    sys.exit(1)

# ============================================================
# 第3部分：构造训练数据结构
# ============================================================

print("="*70)
print("🔧 第三步：构造训练数据")
print("="*70 + "\n")

training_data = []
for instruction_item, data_item in zip(all_instructions, all_data):
    sample = {
        'instruction': instruction_item.get('instruction', ''),
        'output': data_item,
        'metadata': {
            'file_id': instruction_item.get('file_id', ''),
            'step_index': instruction_item.get('step_index', 0),
            'keywords': instruction_item.get('keywords', []),
            'avg_weight': instruction_item.get('keyword_weights', {}).get('avg_weight', 1.0),
        }
    }
    training_data.append(sample)

print(f"✅ 已构造: {len(training_data):,} 个训练样本\n")

# ============================================================
# 第4部分：分割Train/Val（按file_id，防止数据泄漏）
# ============================================================

print("="*70)
print("📊 第四步：分割数据集")
print("="*70 + "\n")

# 按file_id分组
file_id_map = {}
for idx, sample in enumerate(training_data):
    file_id = sample['metadata']['file_id']
    if file_id not in file_id_map:
        file_id_map[file_id] = []
    file_id_map[file_id].append(idx)

# 随机分割file_id（不是样本）
random.seed(42)
all_file_ids = list(file_id_map.keys())
random.shuffle(all_file_ids)

split_point = int(len(all_file_ids) * 0.9)
train_file_ids = set(all_file_ids[:split_point])

# 按file_id分割数据
train_data = []
val_data = []
for file_id, indices in file_id_map.items():
    if file_id in train_file_ids:
        train_data.extend([training_data[i] for i in indices])
    else:
        val_data.extend([training_data[i] for i in indices])

# ============================================================
# 第4.5部分：采样25%数据（OOM优化：从50%降到25%）
# ============================================================

print("="*70)
print("📊 采样25%数据（OOM优化版）")
print("="*70 + "\n")

# 原始数据大小
orig_train_size = len(train_data)
orig_val_size = len(val_data)

# ⚠️ 激进的采样：从50%降到25%以节省内存
print("⚠️  采样比例已从50%降到25%以防止OOM")
print(f"   原始训练集: {orig_train_size:,} 样本")
print(f"   原始验证集: {orig_val_size:,} 样本\n")

random.seed(42)

# 采样25%
sample_indices_train = random.sample(range(len(train_data)), max(1, int(len(train_data) * 0.25)))
sample_indices_val = random.sample(range(len(val_data)), max(1, int(len(val_data) * 0.25)))

train_data = [train_data[i] for i in sorted(sample_indices_train)]
val_data = [val_data[i] for i in sorted(sample_indices_val)]

print(f"   🔄 训练集: {orig_train_size:,} → {len(train_data):,} 样本 (保留25%)")
print(f"   🔄 验证集: {orig_val_size:,} → {len(val_data):,} 样本 (保留25%)\n")

print(f"   ✅ 采样后训练集: {len(train_data):,} 样本")
print(f"   ✅ 采样后验证集: {len(val_data):,} 样本")
print(f"   ✅ 比例: {len(train_data)/(len(train_data)+len(val_data))*100:.1f}% 训练 / {len(val_data)/(len(train_data)+len(val_data))*100:.1f}% 验证\n")

# ============================================================
# 第5部分：数据质量检查
# ============================================================

print("="*70)
print("✅ 数据质量检查")
print("="*70 + "\n")

# 检查完整性
total = len(train_data)
has_instruction = sum(1 for s in train_data if 'instruction' in s and s['instruction'])
has_output = sum(1 for s in train_data if 'output' in s)
has_keywords = sum(1 for s in train_data if s.get('metadata', {}).get('keywords'))

print(f"   训练集完整性:")
print(f"      指令完整: {has_instruction}/{total} ({has_instruction/total*100:.1f}%)")
print(f"      输出完整: {has_output}/{total} ({has_output/total*100:.1f}%)")
print(f"      关键词完整: {has_keywords}/{total} ({has_keywords/total*100:.1f}%)\n")

# 显示示例
if train_data:
    sample = train_data[0]
    print(f"📝 数据样本:")
    print(f"   指令: {sample.get('instruction', '')[:80]}...")
    print(f"   输出字段: {list(sample.get('output', {}).keys())}")
    print(f"   关键词: {sample.get('metadata', {}).get('keywords', [])[:3]}\n")

print("="*70)
print("✅ 数据加载、采样和分割完成！")
print("="*70)
print("\n📊 可用变量:")
print("   • train_data: 训练集数据 (已采样25%)")
print("   • val_data: 验证集数据 (已采样25%)")
print(f"   • 总样本数: {len(train_data) + len(val_data):,} (原始: {orig_train_size + orig_val_size:,})")
print()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, TaskType, get_peft_model

# ============================================================
# 模型配置
# ============================================================

MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"  # 无需认证、专为代码优化
OUTPUT_DIR = "/content/drive/MyDrive/gis-models/step-level-model"
LORA_R = 32
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
BATCH_SIZE = 1  # ⚠️ 从2降到1（激进内存优化）
GRADIENT_ACCUMULATION = 4  # 用梯度积累来补偿batch_size的减少，相当于batch=4
EVAL_AND_SAVE_STEPS = 50
MAX_LENGTH = 192  # ⚠️ 从256降到192（节省20%的内存）

print("🔧 导入完成，模型配置已准备")
print(f"   • MODEL_NAME: {MODEL_NAME}")
print(f"   • BATCH_SIZE: {BATCH_SIZE}")
print(f"   • GRADIENT_ACCUMULATION: {GRADIENT_ACCUMULATION}")
print(f"   • 有效batch大小: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"   • MAX_LENGTH: {MAX_LENGTH}")
print(f"   • LORA_R: {LORA_R}")
print(f"   • OUTPUT_DIR: {OUTPUT_DIR}\n")
print("="*70)
print("🔧 步骤5：训练配置（OOM优化版）")
print("="*70 + "\n")

# ============================================================
# 训练参数配置
# ============================================================

# 内存优化配置
BATCH_SIZE = 1  # 单个样本batch（激进）
GRADIENT_ACCUMULATION_STEPS = 4  # 每4步累积梯度（相当于batch_size=4）
MAX_LENGTH = 192  # 更短的序列长度

# 学习率和优化器配置
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 3
WARMUP_STEPS = int(len(train_data) / (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS) * 0.1)  # 10% warmup
MAX_STEPS = -1  # 使用epochs而不是steps限制

print(f"📊 批处理配置（OOM优化）:")
print(f"   • 单个batch_size: {BATCH_SIZE}")
print(f"   • GRADIENT_ACCUMULATION_STEPS: {GRADIENT_ACCUMULATION_STEPS}")
print(f"   • 有效batch大小: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"   • 每个epoch的优化步数: {len(train_data) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS):,}\n")

print(f"📈 优化器配置:")
print(f"   • LEARNING_RATE: {LEARNING_RATE}")
print(f"   • WEIGHT_DECAY: {WEIGHT_DECAY}")
print(f"   • WARMUP_STEPS: {WARMUP_STEPS:,}\n")

print(f"🔄 训练周期配置:")
print(f"   • NUM_EPOCHS: {NUM_EPOCHS}")
print(f"   • MAX_LENGTH: {MAX_LENGTH} tokens (降低20%)")
print(f"   • TRAINING_SAMPLES: {len(train_data):,}")
print(f"   • VALIDATION_SAMPLES: {len(val_data):,}\n")

# ============================================================
# 估算训练时间
# ============================================================

steps_per_epoch = len(train_data) / (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
total_steps = steps_per_epoch * NUM_EPOCHS

print(f"⏱️  训练时间估算 (T4 GPU):")
print(f"   • 总优化步数: {total_steps:.0f}")
print(f"   • 每个epoch: {steps_per_epoch:.0f} 优化步")
print(f"   • 预计时间: 2-3 小时 (20k样本, batch_size=1+GradAcc=4, 3 epochs)")
print(f"   (相比于原始配置，时间略长但更稳定)\n")

# ============================================================
# 显示内存优化配置
# ============================================================

print("💾 内存优化设置总结:")
print("   ✓ float16 精度")
print("   ✓ Gradient Checkpointing (已启用)")
print("   ✓ AdamW优化器（标准版本）")
print("   ✓ BATCH_SIZE=1 + GRAD_ACC=4 (激进内存节省)")
print("   ✓ MAX_LENGTH=192 (减少20%)")
print("   ✓ 50% 数据采样 (~20k样本)")
print("   ✓ 优化的Tokenization批处理")
print("\n   预期显存使用: ~8-10 GB (in T4's 14GB)\n")

print("="*70)
print("🚀 准备就绪！下一步：运行训练")
print("="*70)
print()

## 🚀 步骤4：模型加载与LoRA配置

In [ ]:
# 升级库 - 确保版本兼容（PEFT版本修复）
print("🔧 Fixing PEFT version compatibility issue...")
print("   ⚠️  卸载不兼容的PEFT版本...\n")

# 强制卸载旧PEFT
!pip uninstall -y peft 2>&1 | head -5

print("\n   ⚠️  安装兼容的PEFT版本...\n")

# 重新安装兼容的版本
!pip install -q peft==0.11.1 --no-cache-dir
!pip install -q --upgrade transformers==4.46.0 --no-cache-dir
!pip install -q --upgrade accelerate>=1.0.0 --no-cache-dir
!pip install -q --upgrade "bitsandbytes>=0.43.0" --no-cache-dir

print("\n✅ Libraries fixed and upgraded!")
print("   • peft==0.11.1 (downgraded to fix ArrowConfig import)")
print("   • transformers==4.46.0")
print("   • accelerate>=1.0.0")
print("   • bitsandbytes>=0.43.0")

In [ ]:
# ⚠️ 可选：超激进OOM恢复方案 - 4bit量化
# 如果运行到这里时仍然OOM，取消注释下面的部分并设置USE_4BIT=True

USE_4BIT = False  # 仅在持续OOM时改为True

if USE_4BIT:
    print("⚠️  启用4-bit量化（超激进内存节省）...")
    from transformers import BitsAndBytesConfig
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    print("   ❌ 停用此cell中的标准加载，下一个cell中的model加载会自动使用4-bit配置")
else:
    print("✅ 使用标准float16加载（内存高效但不是超激进）")
    print("   如果出现OOM，改动此cell中 USE_4BIT = True")


In [ ]:
# 加载tokenizer (CodeLlama)
import gc
import sys

print("📖 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    padding_side="right"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded: vocab_size={len(tokenizer)}")

# ⚠️ 关键：在模型加载前清理内存
print("\n🧹 清理缓存...")
gc.collect()
torch.cuda.empty_cache()

# 清理peft相关的模块缓存（修复ArrowConfig导入问题）
peft_modules = [name for name in list(sys.modules.keys()) if 'peft' in name.lower()]
for module_name in peft_modules:
    try:
        del sys.modules[module_name]
    except:
        pass
print(f"   ✅ 清理了{len(peft_modules)}个peft模块缓存")

# 加载模型 - 简化方案：使用float16而不是8-bit
print("\n🤖 Loading model with float16 precision...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",
)

# 训练中必须禁用缓存以配合梯度检查点
model.config.use_cache = False

# 启用梯度检查点（节省显存）
model.gradient_checkpointing_enable()

print("✅ Base model loaded (float16, ~6-7GB RAM)")

# ⚠️ 再次清理缓存，确保peft导入正确
gc.collect()

# 应用LoRA
print("\n🔧 Applying LoRA...")

from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA applied!")

# 再次清理内存
gc.collect()
torch.cuda.empty_cache()
print("\n💾 GPU Memory Status:")
print(f"   Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

## 📚 步骤5：数据准备与Tokenization

### 🎯 三部分数据结构 (Input/Output/Metadata)

每个训练样本包含：

```
📝 INPUT: 自然语言指令
   例: "Open E MS Kabel"
   来源: step_level_instructions.jsonl → instruction 字段
   
📤 OUTPUT: 完整的结构化步骤数据 (JSON格式)
   例: 
   {
     "step_index": 0,
     "database": ":elektra",
     "object": "E MS Kabel",
     "object_id": "Passed",
     "module": "Editor(s)",
     "method": "Open Object",
     "command": "Execute Object Control Testcommand",
     "test_data": {...},
     "file_id": "file_000001"
   }
   来源: step_level_data.jsonl → 完整的该行数据
   
📊 METADATA: 权重信息和标识符
   例:
   {
     "file_id": "file_000001",
     "step_index": 0,
     "keyword_weights": {
       "keywords": [["Open", 3.0], ["E MS Kabel", 2.0]],
       "avg_weight": 2.5,
       "max_weight": 3.0,
       "keyword_count": 2
     }
   }
   来源: step_level_instructions.jsonl → keyword_weights + file_id
```

### 🚀 相比之前的改进

| 维度 | 之前 | 现在 |
|------|------|------|
| **输入** | instruction + context | instruction (自然语言) |
| **输出** | instruction (重复) | 完整JSON结构化数据 |
| **监督信号** | 弱 | 强（精确的method/object/database等字段） |
| **可用性** | 只是指令重复 | 输出可直接用于API调用/数据库操作 |
| **权重** | 只在元数据中 | 元数据中保留，用于加权损失 |

### 📖 训练流程

```
Input: "Open E MS Kabel"
         ↓
[Tokenization]
         ↓
Model learns to output:
{
  "step_index": 0,
  "object": "E MS Kabel",
  "method": "Open Object",
  ...
}
         ↓
Loss = weighted_loss(pred, target, weight=avg_weight)
         ↓
Validation: JSON validity + field accuracy
```

### 📊 数据统计信息

- **训练样本**: ~36k (来自4013个工作流的90%)
- **验证样本**: ~4k (来自4013个工作流的10%)
- **按file_id分割**: 确保同一文件的所有步骤要么全在train要么全在val
- **权重分布**: avg_weight范围 1.0~3.0，用于调整梯度权重

In [ ]:
from datasets import Dataset
import json as json_module
import gc

# 准备数据集
print("📊 准备datasets（OOM优化版）...\n")

# 转换为Dataset格式
train_dataset_hf = Dataset.from_dict({
    'input': [d['instruction'] for d in train_data],
    'output': [json_module.dumps(d['output'], ensure_ascii=False, indent=2) for d in train_data],
    'avg_weight': [d['metadata']['avg_weight'] for d in train_data],
})

eval_dataset_hf = Dataset.from_dict({
    'input': [d['instruction'] for d in val_data],
    'output': [json_module.dumps(d['output'], ensure_ascii=False, indent=2) for d in val_data],
    'avg_weight': [d['metadata']['avg_weight'] for d in val_data],
})

print(f"  训练集: {len(train_dataset_hf)} samples")
print(f"  验证集: {len(eval_dataset_hf)} samples\n")

# 清理原始数据以节省内存
del train_data
del val_data
gc.collect()
print("✅ 原始train_data/val_data已删除，内存已清理\n")

# 格式化prompt：输入指令 -> 输出完整的步骤数据
def format_prompt(example):
    """
    格式化为prompt：指令 -> 步骤数据
    
    示例：
    Input: "Open E MS Kabel"
    Output: 
    {
      "step_index": 0,
      "database": ":elektra",
      ...
    }
    """
    input_instruction = example['input']
    output_json = example['output']  # 已是格式化的JSON字符串
    
    prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Instruction: {input_instruction}

Step Data JSON:
{output_json}"""
    
    return {"text": prompt}

print("🔄 格式化prompt...")
train_dataset_hf = train_dataset_hf.map(
    format_prompt, 
    remove_columns=['input', 'output', 'avg_weight'],
    batched=False,
    desc="Formatting train"
)

eval_dataset_hf = eval_dataset_hf.map(
    format_prompt, 
    remove_columns=['input', 'output', 'avg_weight'],
    batched=False,
    desc="Formatting val"
)

gc.collect()
print("✅ Prompt格式化完成\n")

# Tokenize - 使用小batch大小来减少内存
def tokenize_function(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,  # 不在这里padding，在collator中padding
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("🔄 Tokenizing（使用小batch大小）...")
# ⚠️ 关键：使用batch_size=512来处理tokenization，而不是默认的大batch
train_dataset_hf = train_dataset_hf.map(
    tokenize_function,
    batched=True,
    batch_size=512,  # 小batch处理，防止OOM
    remove_columns=train_dataset_hf.column_names,
    desc="Tokenizing train"
)

eval_dataset_hf = eval_dataset_hf.map(
    tokenize_function,
    batched=True,
    batch_size=512,
    remove_columns=eval_dataset_hf.column_names,
    desc="Tokenizing val"
)

gc.collect()
print("✅ 数据准备完成！\n")

# 数据统计
print(f"📊 数据统计:")
train_lengths = [len(d['input_ids']) for d in train_dataset_hf]
eval_lengths = [len(d['input_ids']) for d in eval_dataset_hf]
print(f"   训练集: {len(train_dataset_hf)} samples")
print(f"      平均长度: {np.mean(train_lengths):.0f} tokens")
print(f"      最大长度: {max(train_lengths)} tokens")
print(f"   验证集: {len(eval_dataset_hf)} samples")
print(f"      平均长度: {np.mean(eval_lengths):.0f} tokens")
print(f"      最大长度: {max(eval_lengths)} tokens\n")
print(f"   格式: Instruction → Step Data JSON\n")

## 🎯 步骤6：训练

In [ ]:
# 配置训练（OOM优化版本）
print("⚙️ 配置训练（OOM优化版）...")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=10,
    save_steps=EVAL_AND_SAVE_STEPS,
    eval_steps=EVAL_AND_SAVE_STEPS,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    fp16=True,
    bf16=False,
    optim="adamw_torch",  # 标准AdamW而不是8-bit
    lr_scheduler_type="cosine",
    save_total_limit=2,
    report_to="none",
    logging_dir=f"{OUTPUT_DIR}/logs",
    remove_unused_columns=False,
    push_to_hub=False,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    # ⚠️ OOM优化设置
    dataloader_pin_memory=False,  # 降低内存压力
    dataloader_drop_last=True,  # 丢弃不完整的batch
    seed=42,
)

# 使用不进行dynamic padding的data collator来节省内存
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # 因为我们做的是causal LM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_hf,
    eval_dataset=eval_dataset_hf,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("✅ Trainer ready！")
print("\n⚠️  内存优化摘要:")
print("   • BATCH_SIZE=1, GRAD_ACC=4 (显著降低单步显存)")
print("   • MAX_LENGTH=192 (减少token数)")
print("   • 采样25%数据 (从40k→10k)")
print("   • dataloader_pin_memory=False")
print("   • dataloader_drop_last=True")
print("   • 预期显存: 8-10 GB (T4 14GB内)")


In [ ]:
print("\n" + "="*70)
print("🚀 开始训练（OOM优化版，预计2-3小时）...")
print("="*70 + "\n")

# ⚠️ 训练前内存清理（关键）
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("🧹 内存已清理")
print(f"   Current GPU Memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"   Max GPU Memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB\n")

# 开始训练
try:
    trainer.train()
except torch.cuda.OutOfMemoryError as e:
    print("\n❌ CUDA OOM错误！")
    print("   解决方案：")
    print("   1. 重启kernel (Runtime → Restart runtime)")
    print("   2. 进一步减少采样率（改为10%）")
    print("   3. 减少MAX_LENGTH到128")
    print("   4. 考虑使用 load_in_4bit=True 进行4bit量化")
    raise

print("\n" + "="*70)
print("🎉 训练完成！")
print("="*70)

# 训练后内存清理
gc.collect()
torch.cuda.empty_cache()

## 🧪 步骤7：验证与测试

In [ ]:
# 快速测试
import textwrap
import json as json_module

print("🧪 测试模型推理...")

model.eval()

# 从验证集随机采样
test_samples = val_data[:5]

for idx, sample in enumerate(test_samples):
    input_text = sample['input']
    output_data = sample['output']
    avg_weight = sample['metadata']['avg_weight']
    
    # 格式化输出为JSON字符串
    output_json = json_module.dumps(output_data, ensure_ascii=False, indent=2)
    
    prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Instruction: {input_text}

Step Data JSON:
{output_json}"""
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_instruction = generated.split("Instruction: ")[-1].strip()
    
    print(f"\n{'='*70}")
    print(f"Sample #{idx+1} (weight: {avg_weight:.2f})")
    print(f"Input Instruction: {input_text}")
    print(f"\nGenerated Output:")
    print(generated_instruction)
    print(f"\nExpected Output: {output_json[:200]}...")

In [ ]:
# 保存模型
print("💾 Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

# 保存训练信息
import json
training_info = {
    "model_name": MODEL_NAME,
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "train_samples": len(train_dataset_hf),
    "val_samples": len(eval_dataset_hf),
    "data_level": "step_level",
    "weighted_loss": True,
    "source_file": "step_level_instructions.jsonl",
    "date": "2026-02-24",
}

with open(f"{OUTPUT_DIR}/training_info.json", 'w') as f:
    json.dump(training_info, f, indent=2)

print("\n📊 训练总结:")
for key, value in training_info.items():
    print(f"  {key}: {value}")